### messages

In [ ]:
# Each message carries a ROLE telling the model who said what (NOTES.md §8):
#   SystemMessage = instructions/persona/rules (highest authority)
#   HumanMessage  = user input
# The model treats the SystemMessage as the rules and answers the HumanMessage within them.
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage, HumanMessage

model=init_chat_model("ollama:qwen3:8b")

messages=[
    SystemMessage("You are an expert in JSON who only replies in JSON and nothing else"),
    HumanMessage("Write a simple json")
]

reponse=model.invoke(messages)
reponse.content

In [ ]:
# Hand-constructing the two tool-cycle message types (NOTES.md §8):
#   AIMessage  carries tool_calls (the model's request to run a tool)
#   ToolMessage carries the RESULT, linked back by tool_call_id
# Here we fake both (as if a tool had already run) and let the model write the reply.
from langchain_core.messages import content
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

model=init_chat_model("ollama:qwen3:8b")

ai_msg=AIMessage(
    content=[],
    tool_calls=[{
        "name":"get_weather",
        "args":{"location":"China"},
        "id":"call_12"          # the id the ToolMessage must reference
    }]
)

weather_result="Very Sunny"
tool_msg=ToolMessage(
    content=weather_result,
    tool_call_id="call_12"      # links this result to the AIMessage's tool_call above
)

messages=[
    SystemMessage("What is weather in China?"),
    ai_msg,
    tool_msg,
]

reponse=model.invoke(messages)
reponse.content